hshhskjd

In [37]:
import os
import io
import sys
import shutil
from styles import CSS
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import subprocess
from IPython.display import Markdown, display

In [ ]:
load_dotenv(override=True)

openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:6]}")
else:
    print("OpenRouter API Key not set (and this is optional)")


In [39]:
openai = OpenAI()

ollama_url = "http://localhost:11434/v1"
openrouter_url = "https://openrouter.ai/api/v1"

ollama = OpenAI(api_key="ollama", base_url=ollama_url)
openrouter = OpenAI(api_key=openrouter_api_key, base_url=openrouter_url)

In [40]:
models = ["gpt-5-nano", "claude-haiku-4-5", "grok-4-fast-non-reasoning", "gemini-2.5-flash-lite", ]

clients = {"gpt-5-nano": openrouter, "qwen2.5-coder": ollama, "deepseek-coder-v2": ollama, "gpt-oss:20b": ollama, "qwen/qwen3-coder-30b-a3b-instruct": openrouter}


In [ ]:
# Extended: 10 more open-source models from OpenRouter (https://openrouter.ai/models)
OPENROUTER_OPEN_SOURCE_MODELS = [
    "meta-llama/llama-3.1-8b-instruct",
    "meta-llama/llama-3.3-70b-instruct",
    "mistralai/mistral-7b-instruct",
    "google/gemma-2-9b-it",
    "microsoft/phi-3-mini-4k-instruct",
    "qwen/qwen-2.5-7b-instruct",
    "huggingfaceh4/zephyr-7b-beta",
    "openchat/openchat-7b",
    "nousresearch/nous-hermes-2-mixtral-8x7b-dpo",
    "teknium/openhermes-2.5-mistral-7b",
]
for m in OPENROUTER_OPEN_SOURCE_MODELS:
    if m not in models:
        models.append(m)
    clients[m] = openrouter
print(f"Extended to {len(models)} models (added 10 open-source from OpenRouter).")

In [ ]:
# Add week4 to path so system_info can be imported (system_info.py lives in week4/)
_week4 = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if not os.path.isfile(os.path.join(_week4, "system_info.py")):
    _week4 = os.path.abspath(os.path.join(os.getcwd(), "week4"))
if os.path.isfile(os.path.join(_week4, "system_info.py")):
    sys.path.insert(0, _week4)

from system_info import retrieve_system_info, rust_toolchain_info

sys_info = retrieve_system_info()
sys_info


In [43]:
if sys.platform == "win32":
    exe = "main.exe"
    compiler = shutil.which("g++")
    if compiler:
        compile_command = [compiler, "-std=c++17", "-O3", "-DNDEBUG", "main.cpp", "-o", exe]
        run_command = [exe]
    else:
        compile_command = ["g++", "-std=c++17", "-O3", "-DNDEBUG", "main.cpp", "-o", exe]
        run_command = [exe]
else:
    compile_command = ["clang++", "-std=c++17", "-O3", "-mcpu=native", "-flto=thin", "-fvisibility=hidden", "-DNDEBUG", "main.cpp", "-o", "main"]
    run_command = ["./main"]

In [44]:
# Extended: Multi-language port (Java, C#, Scala, JavaScript) in addition to C++
LANGUAGE_OPTIONS = ["C++", "Java", "C#", "Scala", "JavaScript"]

def get_lang_config(lang):
    """Config per target language: extension, main file, compile/run commands."""
    csharp_compile = ["csc", "/out:main.exe", "main.cs"] if shutil.which("csc") else (["dotnet", "build", "main.csproj"] if os.path.isfile("main.csproj") else None)
    csharp_run = ["main.exe"] if sys.platform == "win32" else ["./main.exe"]
    base = {
        "C++": {"ext": "cpp", "file": "main.cpp", "compile": compile_command, "run": run_command},
        "Java": {"ext": "java", "file": "Main.java", "compile": ["javac", "Main.java"], "run": ["java", "Main"]},
        "C#": {"ext": "cs", "file": "main.cs", "compile": csharp_compile, "run": csharp_run},
        "Scala": {"ext": "scala", "file": "Main.scala", "compile": ["scalac", "Main.scala"], "run": ["scala", "Main"]},
        "JavaScript": {"ext": "js", "file": "main.js", "compile": None, "run": ["node", "main.js"]},
    }
    return base.get(lang, base["C++"])

In [45]:
default_python_code = f"""
import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations+1):
        j = i * param1 - param2
        result -= (1/j)
        j = i * param1 + param2
        result += (1/j)
    return result

start_time = time.time()
result = calculate(200_000_000, 4, 1) * 4
end_time = time.time()

print(f"Result: {{result:.12f}}")
print(f"Execution Time: {{(end_time - start_time):.6f}} seconds")
"""

In [46]:
language = "C++"
extension = "rs" if language == "Rust" else "cpp"


system_prompt = """
Your task is to convert Python code into high performance C++ code.
Respond only with C++ code. Do not provide any explanation other than occasional comments.
The C++ response needs to produce an identical output in the fastest possible time.
"""

def user_prompt_for(python):
    return f"""
Port this Python code to C++ with the fastest possible implementation that produces identical output in the least time.
The system information is:
{sys_info}
Your response will be written to a file called main.cpp and then compiled and executed; the compilation command is:
{compile_command}
Respond only with C++ code.
Python code to port:

```python
{python}
```
"""

In [47]:
# Extended: prompts and write/run per language
def get_system_prompt_for_lang(lang):
    if lang == "C++":
        return system_prompt
    return f"""Your task is to convert Python code into high-performance {lang} code.
Respond only with {lang} code. Do not provide any explanation other than occasional comments.
The {lang} code must produce identical output to the Python code. Use standard libraries only."""

def get_user_prompt_for_lang(lang, python):
    cfg = get_lang_config(lang)
    comp = cfg.get("compile") or "N/A (interpreted)"
    return f"""Port this Python code to {lang} with identical output.
System information: {sys_info}
Compilation/run: {comp}
Respond only with {lang} code. Python code:

```python
{python}
```"""

In [48]:
def messages_for(python):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(python)}
    ]

In [49]:

import re

def _strip_code_blocks(reply, lang):
    """Remove markdown code fences (e.g. ```java ... ```)."""
    aliases = {"C++": ["cpp", "c++"], "Java": ["java"], "C#": ["csharp", "cs", "c#"], "Scala": ["scala"], "JavaScript": ["javascript", "js"]}
    tags = aliases.get(lang, [lang.lower()])
    for tag in tags:
        reply = re.sub(r"```\s*" + re.escape(tag) + r"\s*\n?", "", reply, flags=re.I)
    reply = reply.replace("```", "").strip()
    return reply


In [50]:
def write_output(cpp):
    with open("main.cpp", "w") as f:
        f.write(cpp)

# Extended: write to file for each language
def write_output_for_lang(lang, code):
    cfg = get_lang_config(lang)
    with open(cfg["file"], "w", encoding="utf-8") as f:
        f.write(code)

In [51]:
def port(model, python):
    client = clients[model]
    reasoning_effort = "high" if 'gpt' in model else None
    response = client.chat.completions.create(model=model, messages=messages_for(python), reasoning_effort=reasoning_effort)
    reply = response.choices[0].message.content
    reply = reply.replace('```cpp','').replace('```','')
    write_output(reply)
    return reply

# Extended: port to any supported programming language;
def port_extended(model, language, python):
    if language == "C++":
        return port(model, python)
    client = clients[model]
    reasoning_effort = "high" if "gpt" in model else None
    sys_p = get_system_prompt_for_lang(language)
    user_p = get_user_prompt_for_lang(language, python)
    messages = [{"role": "system", "content": sys_p}, {"role": "user", "content": user_p}]
    response = client.chat.completions.create(model=model, messages=messages, reasoning_effort=reasoning_effort)
    reply = response.choices[0].message.content
    reply = _strip_code_blocks(reply, language)
    write_output_for_lang(language, reply)
    return reply

In [52]:
def run_python(code):
    globals_dict = {"__builtins__": __builtins__}

    buffer = io.StringIO()
    old_stdout = sys.stdout
    sys.stdout = buffer

    try:
        exec(code, globals_dict)
        output = buffer.getvalue()
    except Exception as e:
        output = f"Error: {e}"
    finally:
        sys.stdout = old_stdout

    return output

In [53]:
def compile_and_run(code):
    write_output(code)
    try:
        subprocess.run(compile_command, check=True, text=True, capture_output=True)
        run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)
        return run_result.stdout or ""
    except FileNotFoundError as err:
        return f"An error occurred - C++ compiler not found:\n{err}"

# Extended: compile and run code for the given language
def run_for_lang(lang, code):
    """Compile and run code for the given language. For C++ uses existing compile_and_run."""
    if lang == "C++":
        return compile_and_run(code)
    write_output_for_lang(lang, code)
    cfg = get_lang_config(lang)
    try:
        if cfg.get("compile"):
            subprocess.run(cfg["compile"], check=True, text=True, capture_output=True, cwd=os.getcwd())
        run_result = subprocess.run(cfg["run"], check=True, text=True, capture_output=True, cwd=os.getcwd(), timeout=60)
        return run_result.stdout or ""
    except FileNotFoundError:
        return f"Runtime/compiler not found for {lang}. Install the required toolchain (e.g. JDK for Java, Node for JS, .NET for C#, Scala for Scala) and add to PATH."
    except subprocess.CalledProcessError as e:
        return f"Error:\n{e.stderr or e.stdout or str(e)}"

def convert_click(model, language, python):
    return port(model, python) if language == "C++" else port_extended(model, language, python)

def run_generated(language, code):
    return compile_and_run(code) if language == "C++" else run_for_lang(language, code)

In [59]:
# --- Agentic: code generator tool that writes unit test cases ---
import json

# Default test framework per language
DEFAULT_TEST_FRAMEWORKS = {
    "Python": "pytest",
    "C++": "Google Test",
    "Java": "JUnit 5",
    "C#": "xUnit or NUnit",
    "Scala": "ScalaTest",
    "JavaScript": "Jest",
}

def generate_unit_tests(model, source_code, language, test_framework=None):
    """
    Code generator tool: generate unit test cases for the given source code.
    Returns (success: bool, test_code_or_error: str).
    """
    if not source_code or not source_code.strip():
        return False, "No source code provided."
    framework = test_framework or DEFAULT_TEST_FRAMEWORKS.get(language, "standard")
    client = clients.get(model, openrouter)
    system = f"""You are a unit test generator. Generate unit tests for the provided {language} code.
Use the {framework} testing framework (or the standard for that language if not specified).
Output ONLY the test code, no explanations. Wrap in a single code block with the appropriate language tag."""
    user = f"""Generate unit tests for this {language} code. Use {framework}.\n\n```\n{source_code[:12000]}\n```"""
    try:
        kwargs = {"model": model, "messages": [{"role": "system", "content": system}, {"role": "user", "content": user}]}
        if "gpt" in model:
            kwargs["reasoning_effort"] = "medium"
        response = client.chat.completions.create(**kwargs)
        reply = (response.choices[0].message.content or "").strip()
        for tag in ["python", "java", "cpp", "c++", "javascript", "js", "csharp", "cs", "scala"]:
            reply = re.sub(r"```\s*" + tag + r"\s*\n?", "", reply, flags=re.I)
        reply = reply.replace("```", "").strip()
        return True, reply or "No test code generated."
    except Exception as e:
        return False, f"Error generating tests: {e}"

# OpenAI tool definition for the agent
TOOL_GENERATE_UNIT_TESTS = {
    "type": "function",
    "function": {
        "name": "generate_unit_tests",
        "description": "Generate unit test cases for the given source code in the specified language and optional test framework.",
        "parameters": {
            "type": "object",
            "properties": {
                "source_code": {"type": "string", "description": "The source code to generate unit tests for."},
                "language": {"type": "string", "description": "Language of the code: Python, C++, Java, C#, Scala, or JavaScript."},
                "test_framework": {"type": "string", "description": "Optional. e.g. pytest, JUnit 5, Jest, Google Test, NUnit."},
            },
            "required": ["source_code", "language"],
        },
    },
}

def execute_tool_call(name, arguments, model):
    """Execute a single tool call and return the result string."""
    args = arguments if isinstance(arguments, dict) else json.loads(arguments)
    if name == "generate_unit_tests":
        ok, out = generate_unit_tests(
            model=model,
            source_code=args.get("source_code", ""),
            language=args.get("language", "Python"),
            test_framework=args.get("test_framework"),
        )
        return out
    return f"Unknown tool: {name}"

def run_agent(model, user_message, source_code_for_tools, max_rounds=5):
    """
    Agentic loop: run the model with tool-calling; when it calls generate_unit_tests, execute it and continue.
    source_code_for_tools: code to pass to the tool when the agent asks for tests (e.g. current editor content).
    Returns (final_response_text, generated_test_code or None).
    """
    client = clients.get(model, openrouter)
    tools = [TOOL_GENERATE_UNIT_TESTS]
    messages = [
        {"role": "system", "content": "You are a coding assistant. When the user asks for unit tests, use the generate_unit_tests tool with the source code they refer to (or the code provided in context). Reply concisely and include the generated test code in your answer or confirm it was generated."},
        {"role": "user", "content": user_message + (f"\n\n[Current code in editor for context:\n{source_code_for_tools[:8000]}]" if source_code_for_tools else "")},
    ]
    generated_tests = None
    for _ in range(max_rounds):
        kwargs = {"model": model, "messages": messages, "tools": tools}
        if "gpt" in model:
            kwargs["reasoning_effort"] = "low"
        response = client.chat.completions.create(**kwargs)
        msg = response.choices[0].message
        if not getattr(msg, "tool_calls", None):
            return (msg.content or "").strip(), generated_tests
        # Append assistant message as dict for API
        assistant_msg = {"role": "assistant", "content": getattr(msg, "content", None) or ""}
        if getattr(msg, "tool_calls", None):
            assistant_msg["tool_calls"] = [{"id": t.id, "type": "function", "function": {"name": t.function.name, "arguments": t.function.arguments}} for t in msg.tool_calls]
        messages.append(assistant_msg)
        for tc in msg.tool_calls:
            name = tc.function.name
            args = json.loads(tc.function.arguments or "{}")
            if name == "generate_unit_tests" and source_code_for_tools:
                args["source_code"] = args.get("source_code") or source_code_for_tools
            result = execute_tool_call(name, args, model)
            if name == "generate_unit_tests" and isinstance(result, str) and "Error" not in result[:50]:
                generated_tests = result
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})
    return (messages[-1].get("content", "Max rounds reached.") if isinstance(messages[-1], dict) else "Max rounds reached.", generated_tests)

def agentic_generate_click(model, source_code, language, test_framework):
    """UI: direct call to the unit test generator tool."""
    ok, out = generate_unit_tests(model, source_code, language, test_framework or None)
    return out

def next_action(model, language, ported_code, action_text):
    """Run a user-specified action on the ported code (e.g. generate test cases, explain, add comments)."""
    if not (action_text or "").strip():
        return "Enter an action (e.g. generate test cases, explain the loop, add comments)."
    if not (ported_code or "").strip():
        return "Port the code first, then type your next action."
    client = clients.get(model, openrouter)
    system = f"""You are a coding assistant. The user has {language} code below and asks you to do something (e.g. generate test cases, explain a part, add comments). Fulfill the request clearly. If generating code, output it in a code block."""
    user = f"Ported {language} code:\n\n```\n{ported_code[:14000]}\n```\n\nUser request: {action_text.strip()}"
    try:
        kwargs = {"model": model, "messages": [{"role": "system", "content": system}, {"role": "user", "content": user}]}
        if "gpt" in model:
            kwargs["reasoning_effort"] = "medium"
        response = client.chat.completions.create(**kwargs)
        return (response.choices[0].message.content or "").strip()
    except Exception as e:
        return f"Error: {e}"

def agent_chat_submit(history, message, model, source_code):
    """UI: agent chat; run_agent returns (reply, generated_tests)."""
    if not (message or "").strip():
        return history, "", None
    reply, tests = run_agent(model, (message or "").strip(), source_code or "")
    new_history = list(history) + [(message, reply)]
    return new_history, "", (tests or "")

In [60]:
# Force Gradio dark mode
force_dark_mode = """
function refresh() {
    const url = new URL(window.location);
    if (url.searchParams.get('__theme') !== 'dark') {
        url.searchParams.set('__theme', 'dark');
        window.location.href = url.href;
    }
}"""

In [ ]:
# Default Gradio UI
with gr.Blocks(css=CSS, theme=gr.themes.Monochrome(), title=f"Port from Python to {language}", js=force_dark_mode) as ui:
    with gr.Row(equal_height=True):
        with gr.Column(scale=6):
            python = gr.Code(
                label="Python (original)",
                value=default_python_code,
                language="python",
                lines=26
            )
        with gr.Column(scale=6):
            cpp = gr.Code(
                label=f"{language} (generated)",
                value="",
                language="cpp",
                lines=26
            )

    with gr.Row(elem_classes=["controls"]):
        python_run = gr.Button("Run Python", elem_classes=["run-btn", "py"])
        model = gr.Dropdown(models, value=models[0], show_label=False)
        convert = gr.Button(f"Port to {language}", elem_classes=["convert-btn"])
        cpp_run = gr.Button(f"Run {language}", elem_classes=["run-btn", "cpp"])

    with gr.Row(equal_height=True):
        with gr.Column(scale=6):
            python_out = gr.TextArea(label="Python result", lines=8, elem_classes=["py-out"])
        with gr.Column(scale=6):
            cpp_out = gr.TextArea(label=f"{language} result", lines=8, elem_classes=["cpp-out"])

    convert.click(fn=port, inputs=[model, python], outputs=[cpp])
    python_run.click(fn=run_python, inputs=[python], outputs=[python_out])
    cpp_run.click(fn=compile_and_run, inputs=[cpp], outputs=[cpp_out])

ui.launch(inbrowser=True)

In [ ]:
# Extended Gradio UI with multi-language port
with gr.Blocks(css=CSS, theme=gr.themes.Monochrome(), title=f"Port from Python to C++ / Java / C# / Scala / JavaScript", js=force_dark_mode) as ui:
    with gr.Row(equal_height=True):
        with gr.Column(scale=6):
            python = gr.Code(
                label="Python (original)",
                value=default_python_code,
                language="python",
                lines=26
            )
        with gr.Column(scale=6):
            cpp = gr.Code(
                label="Generated code",
                value="",
                language="cpp",
                lines=26
            )

    with gr.Row(elem_classes=["controls"]):
        python_run = gr.Button("Run Python", elem_classes=["run-btn", "py"])
        model = gr.Dropdown(models, value=models[0], show_label=False)
        convert = gr.Button(f"Port to -->", elem_classes=["convert-btn"])
        lang_dropdown = gr.Dropdown(LANGUAGE_OPTIONS, value="C++", show_label=False)
        cpp_run = gr.Button("Run generated code", elem_classes=["run-btn", "cpp"])

    with gr.Row(equal_height=True):
        with gr.Column(scale=6):
            python_out = gr.TextArea(label="Python result", lines=8, elem_classes=["py-out"])
        with gr.Column(scale=6):
            cpp_out = gr.TextArea(label="Run result", lines=8, elem_classes=["cpp-out"])

    convert.click(fn=convert_click, inputs=[model, lang_dropdown, python], outputs=[cpp])
    python_run.click(fn=run_python, inputs=[python], outputs=[python_out])
    cpp_run.click(fn=run_generated, inputs=[lang_dropdown, cpp], outputs=[cpp_out])

ui.launch(inbrowser=True)

In [ ]:
# Extended Gradio UI with Agentic Unit Tests
with gr.Blocks(css=CSS, theme=gr.themes.Monochrome(), title="Port from Python + Agentic Unit Tests", js=force_dark_mode) as ui:
    with gr.Tabs():
        with gr.Tab("Port"):
            with gr.Row(equal_height=True):
                with gr.Column(scale=6):
                    python = gr.Code(
                        label="Python (original)",
                        value=default_python_code,
                        language="python",
                        lines=26
                    )
                with gr.Column(scale=6):
                    cpp = gr.Code(
                        label="Generated code",
                        value="",
                        language="cpp",
                        lines=26
                    )

            with gr.Row(elem_classes=["controls"]):
                python_run = gr.Button("Run Python", elem_classes=["run-btn", "py"])
                model = gr.Dropdown(models, value=models[0], show_label=False)
                convert = gr.Button("Port to -->", elem_classes=["convert-btn"])
                lang_dropdown = gr.Dropdown(LANGUAGE_OPTIONS, value="C++", show_label=False)
                cpp_run = gr.Button("Run generated code", elem_classes=["run-btn", "cpp"])

            with gr.Row():
                next_action_in = gr.Textbox(placeholder="e.g. generate test cases, explain the workflow, add comments...", label="Next action (after port)", scale=4)
                next_action_btn = gr.Button("Do it", elem_classes=["convert-btn"], scale=1)

            with gr.Row(equal_height=True):
                with gr.Column(scale=4):
                    python_out = gr.TextArea(label="Python result", lines=8, elem_classes=["py-out"])
                with gr.Column(scale=4):
                    cpp_out = gr.TextArea(label="Generated code result", lines=8, elem_classes=["cpp-out"])
                with gr.Column(scale=4):
                    next_action_out = gr.TextArea(label="Action result", lines=10, elem_classes=["py-out"])


            convert.click(fn=convert_click, inputs=[model, lang_dropdown, python], outputs=[cpp])
            python_run.click(fn=run_python, inputs=[python], outputs=[python_out])
            cpp_run.click(fn=run_generated, inputs=[lang_dropdown, cpp], outputs=[cpp_out])
            next_action_btn.click(fn=next_action, inputs=[model, lang_dropdown, cpp, next_action_in], outputs=[next_action_out])

        with gr.Tab("Agentic"):
            agent_code = gr.Code(label="Code to generate tests for", value=default_python_code, language="python", lines=16)
            with gr.Row():
                agent_model = gr.Dropdown(models, value=models[0], label="Model")
                agent_lang = gr.Dropdown(LANGUAGE_OPTIONS, value="Python", label="Language")
                agent_fw = gr.Textbox(placeholder="e.g. pytest, JUnit 5 (optional)", label="Test framework")
            agent_gen_btn = gr.Button("Generate unit tests (tool)", elem_classes=["convert-btn"])
            agent_tests_out = gr.Code(label="Generated unit tests", value="", language="python", lines=24)
            agent_chatbot = gr.Chatbot(label="Agent chat (can ask for unit tests)", height=280)
            agent_msg = gr.Textbox(placeholder="e.g. Generate unit tests for the code above", label="Message")
            agent_send_btn = gr.Button("Send")
            agent_state = gr.State([])
            agent_gen_btn.click(fn=agentic_generate_click, inputs=[agent_model, agent_code, agent_lang, agent_fw], outputs=[agent_tests_out])
            def agent_chat_fn(history, msg, model, code):
                if not (msg or "").strip(): return history, "", "", history
                reply, tests = run_agent(model, msg.strip(), code)
                new_h = list(history) + [(msg, reply)]
                return new_h, "", tests or "", new_h
            agent_send_btn.click(fn=agent_chat_fn, inputs=[agent_state, agent_msg, agent_model, agent_code], outputs=[agent_chatbot, agent_msg, agent_tests_out, agent_state])

ui.launch(inbrowser=True)